# Preprocessing for DuckDB WASM

Prepares parquet files for direct browser queries via DuckDB WASM.

**Task B — Pre-aggregate timeseries**  
The timeseries chart needs only `(year, month, vessel_count)` per spatial cell. Aggregating all 5 annual monthly files into one small file sorted by `(lat, lon)` avoids scanning 476 MB on every viewport change.

Output: `data/parquet/timeseries_grid.parquet` (single file, expected 20–80 MB)

**Task C — Split into per-day parquet files**  
HTTP range requests don't work reliably on HuggingFace, so DuckDB WASM downloads the full file. Monthly files are 80–175 MB; splitting into one file per day brings each download to ~3–5 MB.

Output: `data/parquet/daily_split/fleet-daily-YYYY-MM-DD.parquet` (~1 825 files)

**Task D — EEZ grid lookup**  
Pre-compute a mapping from every unique `(cell_ll_lat, cell_ll_lon)` grid cell to its EEZ.

Output: `data/parquet/eez_grid.parquet`

In [1]:
from pathlib import Path
import duckdb
import pyarrow.parquet as pq

DATA_DIR       = Path("../data/parquet")
DAILY_DIR      = DATA_DIR / "daily"
MONTHLY_DIR    = DATA_DIR / "monthly"
TIMESERIES_OUT = DATA_DIR / "timeseries_grid.parquet"

con = duckdb.connect()
print("DuckDB", duckdb.__version__)


def print_rg_stats(path: Path, n_groups: int = 4) -> None:
    meta = pq.read_metadata(path)
    print(f"{path.name}")
    print(f"  {meta.num_row_groups} row groups, {meta.num_rows:,} rows")
    for g in range(min(n_groups, meta.num_row_groups)):
        rg = meta.row_group(g)
        print(f"  row_group[{g}]  ({rg.num_rows:,} rows)")
        for i in range(rg.num_columns):
            col   = rg.column(i)
            stats = col.statistics
            if stats:
                print(f"    {col.path_in_schema:<20} min={str(stats.min):<25} max={stats.max}")
            else:
                print(f"    {col.path_in_schema:<20} (no statistics)")
    print()

DuckDB 1.5.0


## Task B — Pre-aggregate timeseries

Reads all 5 annual monthly files, drops the per-flag/geartype breakdown (not needed by the timeseries chart), aggregates to `(year, month, cell_lat, cell_lon)`, and sorts by `(cell_ll_lat, cell_ll_lon)` so spatial predicate pushdown works on the result file.

In [2]:
if TIMESERIES_OUT.exists():
    print(f"skip  {TIMESERIES_OUT.name} (already exists)")
else:
    monthly_glob = str(MONTHLY_DIR / "*.parquet")

    con.execute(f"""
        COPY (
            SELECT
                year::SMALLINT     AS year,
                month::SMALLINT    AS month,
                cell_ll_lat::FLOAT AS cell_ll_lat,
                cell_ll_lon::FLOAT AS cell_ll_lon,
                SUM(mmsi_present)::INTEGER AS vessel_count
            FROM read_parquet('{monthly_glob}')
            GROUP BY year, month, cell_ll_lat, cell_ll_lon
            ORDER BY cell_ll_lat, cell_ll_lon, year, month
        ) TO '{TIMESERIES_OUT}'
        (FORMAT PARQUET, COMPRESSION SNAPPY, ROW_GROUP_SIZE 50000)
    """)

    size_mb = TIMESERIES_OUT.stat().st_size / 1e6
    meta    = pq.read_metadata(TIMESERIES_OUT)
    print(f"wrote {TIMESERIES_OUT.name}  {size_mb:.1f} MB  ({meta.num_row_groups} row groups, {meta.num_rows:,} rows)")

skip  timeseries_grid.parquet (already exists)


### Verify: timeseries_grid row group statistics

In [3]:
print_rg_stats(TIMESERIES_OUT)

timeseries_grid.parquet
  797 row groups, 40,770,811 rows
  row_group[0]  (51,200 rows)
    year                 min=2020                      max=2024
    month                min=1                         max=12
    cell_ll_lat          min=-77.9000015258789         max=-69.4000015258789
    cell_ll_lon          min=-180.0                    max=179.89999389648438
    vessel_count         min=1                         max=12
  row_group[1]  (51,200 rows)
    year                 min=2020                      max=2024
    month                min=1                         max=12
    cell_ll_lat          min=-69.4000015258789         max=-63.79999923706055
    cell_ll_lon          min=-180.0                    max=179.89999389648438
    vessel_count         min=1                         max=12
  row_group[2]  (51,200 rows)
    year                 min=2020                      max=2024
    month                min=1                         max=12
    cell_ll_lat          min=-63.799999

### Smoke-test: query timeseries_grid for a sample viewport

In [4]:
# South-East Asia viewport
west, east, south, north = 100.0, 130.0, 0.0, 30.0

result = con.execute(f"""
    SELECT year, month, SUM(vessel_count)::INTEGER AS vessel_count
    FROM read_parquet('{TIMESERIES_OUT}')
    WHERE cell_ll_lon BETWEEN {west} AND {east}
      AND cell_ll_lat BETWEEN {south} AND {north}
    GROUP BY year, month
    ORDER BY year, month
""").fetchdf()

print(result.to_string(index=False))

 year  month  vessel_count
 2020      1        193446
 2020      2        155860
 2020      3        271177
 2020      4        318892
 2020      5         84701
 2020      6         88981
 2020      7         92994
 2020      8        304780
 2020      9        325182
 2020     10        266852
 2020     11        265034
 2020     12        253456
 2021      1        292543
 2021      2        309059
 2021      3        316584
 2021      4        339455
 2021      5        122030
 2021      6        140520
 2021      7        132042
 2021      8        319141
 2021      9        361217
 2021     10        307185
 2021     11        122362
 2021     12        685755
 2022      1        581109
 2022      2        397331
 2022      3        458027
 2022      4        491300
 2022      5        212888
 2022      6        233979
 2022      7        244970
 2022      8        623265
 2022      9        753769
 2022     10        701873
 2022     11        930608
 2022     12        716220
 

## Task C — Split into per-day parquet files

HTTP range requests don't work reliably on HuggingFace (redirects break byte-range sub-requests),
so DuckDB WASM always downloads the full parquet file. Monthly files are 80–175 MB; splitting into
one file per day brings each download to ~3–5 MB.

`date` is dropped from each file — it is encoded in the filename and removing it shrinks the files.
Rows are sorted by `(cell_ll_lat, cell_ll_lon)` so spatial predicate pushdown still works within a file.

Output: `data/parquet/daily_split/fleet-daily-YYYY-MM-DD.parquet` (~1 825 files)

In [ ]:
DAILY_SPLIT_DIR = DATA_DIR / "daily_split"
DAILY_SPLIT_DIR.mkdir(exist_ok=True)

monthly_files = sorted(DAILY_DIR.glob("fleet-daily-*.parquet"))
print(f"Found {len(monthly_files)} monthly files")

# GFW UNKNOWN-<ISO> flags are collapsed into <ISO> here (bare UNKNOWN kept):
# invalid-MID vessels spending >50% of fishing hours in the <ISO> EEZ are most
# likely <ISO>-flagged activity broadcasting a wrong MID (README-known-issues-v3.txt).

for src in monthly_files:
    dates = con.execute(
        f"SELECT DISTINCT date FROM read_parquet('{src}') ORDER BY date"
    ).fetchall()

    written = 0
    skipped = 0
    for (date,) in dates:
        date_str = str(date)          # '2020-01-01'
        dst = DAILY_SPLIT_DIR / f"fleet-daily-{date_str}.parquet"
        if dst.exists():
            skipped += 1
            continue
        con.execute(f"""
            COPY (
                SELECT
                    cell_ll_lat,
                    cell_ll_lon,
                    CASE WHEN flag LIKE 'UNKNOWN-_%' THEN flag[9:] ELSE flag END AS flag,
                    geartype, fishing_hours, mmsi_present
                FROM read_parquet('{src}')
                WHERE date = '{date_str}'
                ORDER BY cell_ll_lat, cell_ll_lon
            ) TO '{dst}'
            (FORMAT PARQUET, COMPRESSION SNAPPY, ROW_GROUP_SIZE 10000)
        """)
        written += 1

    tag = f"wrote {written}" if written else "skip all"
    if skipped:
        tag += f"  (skipped {skipped} existing)"
    print(f"  {src.name}  → {tag}")

print("\nDone.")

In [6]:
sample_day = DAILY_SPLIT_DIR / "fleet-daily-2021-03-15.parquet"
meta = pq.read_metadata(sample_day)
size_mb = sample_day.stat().st_size / 1e6

print(f"{sample_day.name}  {size_mb:.2f} MB  ({meta.num_row_groups} row groups, {meta.num_rows:,} rows)")
print()
print_rg_stats(sample_day, n_groups=3)

fleet-daily-2021-03-15.parquet  3.87 MB  (74 row groups, 751,372 rows)

fleet-daily-2021-03-15.parquet
  74 row groups, 751,372 rows
  row_group[0]  (10,240 rows)
    cell_ll_lat          min=-69.0199966430664         max=-49.970001220703125
    cell_ll_lon          min=-135.72000122070312       max=170.3800048828125
    flag                 min=ARG                       max=VUT
    geartype             min=fishing                   max=trawlers
    fishing_hours        min=0.0                       max=8.194700241088867
    mmsi_present         min=1                         max=12
  row_group[1]  (10,240 rows)
    cell_ll_lat          min=-49.970001220703125       max=-46.060001373291016
    cell_ll_lon          min=-76.08000183105469        max=170.8699951171875
    flag                 min=ARG                       max=VUT
    geartype             min=fishing                   max=trawlers
    fishing_hours        min=0.0                       max=21.5049991607666
    mmsi_present  

## Task D — EEZ grid lookup

Pre-compute a mapping from every unique `(cell_ll_lat, cell_ll_lon)` grid cell to its EEZ.
Done once here against the full global shapefile (no region restriction); the result is joined
at query time by DuckDB WASM to power the illegal-fishing lollipop chart.

Output: `data/parquet/eez_grid.parquet`  
Columns: `cell_ll_lat, cell_ll_lon, eez_iso, eez_name, eez_sovereign`

In [ ]:
import geopandas as gpd

EEZ_SHP = Path("../data/World_EEZ_v12_20231025/eez_v12.shp")
EEZ_OUT = DATA_DIR / "eez_grid.parquet"

# The raw data is at 0.01° resolution (~154M distinct cells globally).
# EEZ zones are tens of km wide, so 0.1° resolution is sufficient and
# reduces the grid to ~1.5M cells. The frontend join must round cell
# coordinates to 0.1° before matching (ROUND(lat / 0.1) * 0.1).

GRID_RES = 0.1

if EEZ_OUT.exists():
    print(f"skip  {EEZ_OUT.name} (already exists)")
else:
    # Step 1: distinct 0.1° cells from all daily files
    cells_df = con.execute(f"""
        SELECT DISTINCT
            (ROUND(cell_ll_lat::DOUBLE / {GRID_RES}) * {GRID_RES})::DOUBLE AS cell_ll_lat,
            (ROUND(cell_ll_lon::DOUBLE / {GRID_RES}) * {GRID_RES})::DOUBLE AS cell_ll_lon
        FROM read_parquet('{DAILY_DIR}/*.parquet')
    """).fetchdf()
    print(f"{len(cells_df):,} unique {GRID_RES}° grid cells")

    # Step 2: load EEZ, keep standard 200NM zones only
    eez = gpd.read_file(EEZ_SHP)
    eez = (
        eez[eez["POL_TYPE"] == "200NM"]
        [["GEONAME", "ISO_TER1", "SOVEREIGN1", "geometry"]]
        .rename(columns={"GEONAME": "eez_name", "ISO_TER1": "eez_iso", "SOVEREIGN1": "eez_sovereign"})
        .reset_index(drop=True)
    )
    print(f"{len(eez)} EEZ polygons loaded")

    # Step 3: point-in-polygon join
    points_gdf = gpd.GeoDataFrame(
        cells_df,
        geometry=gpd.points_from_xy(cells_df["cell_ll_lon"], cells_df["cell_ll_lat"]),
        crs="EPSG:4326",
    )
    joined = gpd.sjoin(points_gdf, eez, how="left", predicate="within")

    # A cell can match multiple zones if polygons overlap — keep the first match
    joined = joined.drop_duplicates(subset=["cell_ll_lat", "cell_ll_lon"])

    # Step 4: cells outside any EEZ → High Seas
    joined["eez_iso"]       = joined["eez_iso"].fillna("INT")
    joined["eez_name"]      = joined["eez_name"].fillna("High Seas")
    joined["eez_sovereign"] = joined["eez_sovereign"].fillna("International Waters")

    result = (
        joined[["cell_ll_lat", "cell_ll_lon", "eez_iso", "eez_name", "eez_sovereign"]]
        .reset_index(drop=True)
    )

    result.to_parquet(EEZ_OUT, index=False, compression="snappy")

    size_mb = EEZ_OUT.stat().st_size / 1e6
    print(f"wrote {EEZ_OUT.name}  {size_mb:.1f} MB  ({len(result):,} cells, {result['eez_iso'].nunique()} distinct EEZs)")